# Ingestão — clima real (Coffee_Data_Set.csv)

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/Coffee_Data_Set.csv", parse_dates=["Date"])
df.head()

In [ ]:
partes = df["Location"].str.split("_")
df["pais"] = partes.str[0]
df["regiao"] = partes.str[1:].str.join(" ")
df["temperatura_media"] = (df["Temp_Max"] + df["Temp_Min"]) / 2

clima = df.rename(columns={
    "Precipitation_mm": "precipitacao",
    "Humidity": "umidade",
    "Solar_Radiation": "radiacao_solar",
    "LAT": "latitude",
    "LON": "longitude",
})[["pais", "regiao", "latitude", "longitude", "Date", "temperatura_media", "precipitacao", "umidade", "radiacao_solar"]]

clima.head()

In [ ]:
regioes = clima.groupby(["pais", "regiao"], as_index=False).agg(
    latitude=("latitude", "first"),
    longitude=("longitude", "first"),
)
regioes

## Gravar no Postgres

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from cuid2 import cuid_wrapper

load_dotenv("../.env")
generate_id = cuid_wrapper()
engine = create_engine(os.environ["DATABASE_URL"].split("?")[0])

In [ ]:
with engine.begin() as conn:
    existentes = conn.execute(text('SELECT id, "nomeRegiao", "pais" FROM "Regiao_Produtora"')).fetchall()
    cache = {(r.pais.lower(), r.nomeRegiao.lower()): r.id for r in existentes}
    mapa_id = {}
    for _, linha in regioes.iterrows():
        chave = (linha["pais"].lower(), linha["regiao"].lower())
        if chave in cache:
            regiao_id = cache[chave]
        else:
            regiao_id = generate_id()
            conn.execute(
                text('INSERT INTO "Regiao_Produtora" (id, "nomeRegiao", "pais", "latitude", "longitude") VALUES (:id, :nome, :pais, :lat, :lon)'),
                {"id": regiao_id, "nome": linha["regiao"], "pais": linha["pais"], "lat": linha["latitude"], "lon": linha["longitude"]},
            )
            cache[chave] = regiao_id
        mapa_id[(linha["pais"], linha["regiao"])] = regiao_id

mapa_id

In [ ]:
regioes_map = pd.DataFrame([(p, r, i) for (p, r), i in mapa_id.items()], columns=["pais", "regiao", "regiaoId"])
clima = clima.merge(regioes_map, on=["pais", "regiao"])
clima["id"] = [generate_id() for _ in range(len(clima))]

saida = clima.rename(columns={
    "Date": "data",
    "temperatura_media": "temperaturaMedia",
    "radiacao_solar": "radiacaoSolar",
})[["id", "regiaoId", "data", "temperaturaMedia", "precipitacao", "umidade", "radiacaoSolar"]]

with engine.begin() as conn:
    conn.execute(text('DELETE FROM "Condicao_Climatica" WHERE "regiaoId" = ANY(:ids)'), {"ids": list(set(mapa_id.values()))})
    saida.to_sql("Condicao_Climatica", conn, if_exists="append", index=False, chunksize=5000, method="multi")

len(saida)